# 03-Privacy-Demonstration

##### 1. Load of Raw Dataset

In [12]:
# Loading raw dataset
import json
import pandas as pd

with open("../data/raw_credit_applications.json", "r") as f:
    data = json.load(f)

df = pd.json_normalize(data)
df.head(10)

,_id,spending_behavior,processing_timestamp,applicant_info.full_name,applicant_info.email,applicant_info.ssn,applicant_info.ip_address,applicant_info.gender,applicant_info.date_of_birth,applicant_info.zip_code,...,financials.credit_history_months,financials.debt_to_income,financials.savings_balance,decision.loan_approved,decision.rejection_reason,loan_purpose,decision.interest_rate,decision.approved_amount,financials.annual_salary,notes
0,app_200,"[{'category': 'Shopping', 'amount': 480}, {'ca...",2024-01-15T00:00:00Z,Jerry Smith,jerry.smith17@hotmail.com,596-64-4340,192.168.48.155,Male,2001-03-09,10036,...,23,0.20,31212,False,algorithm_risk_score,NaN,NaN,NaN,NaN,NaN
1,app_037,"[{'category': 'Rent', 'amount': 608}, {'catego...",NaN,Brandon Walker,brandon.walker2@yahoo.com,425-69-4784,10.1.102.112,M,1992-03-31,10032,...,51,0.18,17915,False,algorithm_risk_score,NaN,NaN,NaN,NaN,NaN
2,app_215,"[{'category': 'Rent', 'amount': 109}]",NaN,Scott Moore,scott.moore94@mail.com,370-78-5178,10.240.193.250,Male,1989-10-24,10075,...,41,0.21,37909,True,NaN,vacation,3.7,59000.0,NaN,NaN
3,app_024,"[{'category': 'Fitness', 'amount': 575}]",NaN,Thomas Lee,thomas.lee6@protonmail.com,194-35-1833,192.168.175.67,Male,1983-04-25,10077,...,70,0.35,0,True,NaN,NaN,4.3,34000.0,NaN,NaN
4,app_184,"[{'category': 'Entertainment', 'amount': 463}]",2024-01-15T00:00:00Z,Brian Rodriguez,brian.rodriguez86@aol.com,480-41-2475,172.29.125.105,M,1999-05-21,10080,...,14,0.23,31763,False,algorithm_risk_score,NaN,NaN,NaN,NaN,NaN
5,app_275,"[{'category': 'Entertainment', 'amount': 571}]",NaN,Maria Miller,maria.miller67@outlook.com,417-25-4912,172.25.58.70,F,14/02/1982,10019,...,33,0.05,49933,False,algorithm_risk_score,NaN,NaN,NaN,NaN,NaN
6,app_099,"[{'category': 'Dining', 'amount': 458}]",NaN,Nicholas King,nicholas.king46@outlook.com,613-23-2503,10.62.62.45,Male,28/01/1990,10022,...,61,0.17,30159,True,NaN,NaN,5.6,27000.0,NaN,NaN
7,app_246,"[{'category': 'Healthcare', 'amount': 478}]",NaN,Susan Rivera,susan.rivera74@gmail.com,176-97-1864,192.168.158.59,F,1991-10-11,90223,...,31,0.29,21809,True,NaN,auto,2.8,38000.0,NaN,NaN
8,app_042,"[{'category': 'Insurance', 'amount': 153}, {'c...",NaN,Joseph Lopez,joseph.lopez1@gmail.com,652-70-5530,192.168.91.142,Male,1990-05-04,10044,...,43,0.41,15974,False,algorithm_risk_score,NaN,NaN,NaN,NaN,NaN
9,app_348,"[{'category': 'Fitness', 'amount': 199}, {'cat...",NaN,Michael Mitchell,michael.mitchell42@hotmail.com,100-94-8400,172.28.12.121,Male,1989-10-10,10080,...,5,0.41,13794,False,insufficient_credit_history,NaN,NaN,NaN,NaN,NaN


##### 2. Analyse Governance Problems

##### 2.1. Personal Identifiable Innformation (PII) Identification

In [13]:
possible_pii_patterns = {
    "direct_identifiers": ["name", "email", "ssn", "ip"],
    "indirect_or_special_identifiers": ["birth", "zip", "gender", "behavior"],
    "financial_personal_data": ["salary", "credit", "debt", "savings"]
}

# Create an empty dictionary to store results
detected_pii_by_category = {cat: [] for cat in possible_pii_patterns}

# Fill it
for col in df.columns:
    col_lower = col.lower()
    for category, patterns in possible_pii_patterns.items():
        if any(pattern in col_lower for pattern in patterns):
            detected_pii_by_category[category].append(col)

print("Auto-detected PII fields by category:\n")
for category, cols in detected_pii_by_category.items():
    print(f"{category}:")
    for col in cols:
        print(f"  - {col}")
    print()



Auto-detected PII fields by category:

direct_identifiers:
  - applicant_info.full_name
  - applicant_info.email
  - applicant_info.ssn
  - applicant_info.ip_address
  - applicant_info.zip_code

indirect_or_special_identifiers:
  - spending_behavior
  - applicant_info.gender
  - applicant_info.date_of_birth
  - applicant_info.zip_code

financial_personal_data:
  - financials.credit_history_months
  - financials.debt_to_income
  - financials.savings_balance
  - financials.annual_salary



##### 2.2. Checking how sensitive data is stored

In [14]:
# Checking in which way these variables are stored
print(df['applicant_info.ssn'].head(),
df['applicant_info.full_name'].head(),
df['applicant_info.email'].head(),
df['applicant_info.ip_address'].head(),
df['applicant_info.date_of_birth'].head(),
df[['spending_behavior']].head()
)



0    596-64-4340
1    425-69-4784
2    370-78-5178
3    194-35-1833
4    480-41-2475
Name: applicant_info.ssn, dtype: object 0        Jerry Smith
1     Brandon Walker
2        Scott Moore
3         Thomas Lee
4    Brian Rodriguez
Name: applicant_info.full_name, dtype: object 0     jerry.smith17@hotmail.com
1     brandon.walker2@yahoo.com
2        scott.moore94@mail.com
3    thomas.lee6@protonmail.com
4     brian.rodriguez86@aol.com
Name: applicant_info.email, dtype: object 0    192.168.48.155
1      10.1.102.112
2    10.240.193.250
3    192.168.175.67
4    172.29.125.105
Name: applicant_info.ip_address, dtype: object 0    2001-03-09
1    1992-03-31
2    1989-10-24
3    1983-04-25
4    1999-05-21
Name: applicant_info.date_of_birth, dtype: object                                    spending_behavior
0  [{'category': 'Shopping', 'amount': 480}, {'ca...
1  [{'category': 'Rent', 'amount': 608}, {'catego...
2              [{'category': 'Rent', 'amount': 109}]
3           [{'category': 'Fitnes

In [15]:
df[['financials.savings_balance', 'financials.annual_income']].describe()

,financials.savings_balance
count,502.000000
mean,29493.503984
std,16775.309756
min,-5000.000000
25%,17258.250000
50%,27385.500000
75%,38251.500000
max,88078.000000


The dataset contains raw identifiers like names, emails, SSNs, IP addresses, exact birth dates, and financial data is not encrypted - which are unnecessary for modeling and create high privacy risk. Hashing identifiers, removing unneeded fields, and converting DOB to age help minimize identification and strengthen data protection. We therefore implement:


* Pseudonymization of identifiers (name, email, SSN)
* Removal of unnecessary PII
* Date of birth conversion to age to minimize exposure of sensitive data (Anonymization)

##### 3. Proposal of Privacy Protections Implementations

##### 3.1. Pseudonymize Identifiers - Hashing

In [16]:
import hashlib
import pandas as pd
import pandas as pd
import hashlib
from IPython.display import display


def hash_value(x):
    if pd.isna(x):
        return None
    return hashlib.sha256(str(x).encode()).hexdigest()

# Create hashed columns first
df["full_name_hash"] = df["applicant_info.full_name"].apply(hash_value)
df["email_hash"] = df["applicant_info.email"].apply(hash_value)
df["ssn_hash"] = df["applicant_info.ssn"].apply(hash_value)

# Show before vs after
before_after = df[[
    "applicant_info.full_name", "full_name_hash",
    "applicant_info.email", "email_hash",
    "applicant_info.ssn", "ssn_hash"
]].head(10)

display(before_after)

# Only drop originals after displaying
df.drop(columns=[
    "applicant_info.full_name",
    "applicant_info.email",
    "applicant_info.ssn"
], inplace=True)


,applicant_info.full_name,full_name_hash,applicant_info.email,email_hash,applicant_info.ssn,ssn_hash
0,Jerry Smith,68ee17cf46b0560352c701bbbdb178c01d4cc368014d18...,jerry.smith17@hotmail.com,116648a7761525746032d0ab323ceb01f50d11f7935164...,596-64-4340,2caf30528c21a10e1307b27f9dbbfc312f0c00d46b333e...
1,Brandon Walker,4c539f3c4c8794d5d0b7ef2d2c6f3e5ae3e9ab65f1cb54...,brandon.walker2@yahoo.com,c3522c0b54ef9045c73186bcabb53f8e512360ed17e9cc...,425-69-4784,2f7da45fefdcfb2c5b4f5b6f1465c22054c36e04fc77c1...
2,Scott Moore,4ad1a6eb65ea21350865c8dbae97a882eb82ee36059784...,scott.moore94@mail.com,b299e7d6a37e183bab209eb8df919652117dd16ed16698...,370-78-5178,db120edcee2366a48d6d77c2db8c64c5536b8dc3c3c524...
3,Thomas Lee,0b6a308e6a3e28e0a90c5e4212a34d32c0debeccc37408...,thomas.lee6@protonmail.com,6fbd2478748a29faa143392f28d955133e346d09673963...,194-35-1833,c835719be02018987096d6e49529a24b1d7e7ab35c84b1...
4,Brian Rodriguez,099b981258dca6195cdfe5d42d1d8ab470b12a0926fadc...,brian.rodriguez86@aol.com,f24e7cc1450ee9aa26b833e0593b2420fbfd59b8ad2636...,480-41-2475,41c7de40dc49185886e6ecb37346ec9eabce16087b7508...
5,Maria Miller,e000c1c9876a7cda602fbb5a70afa05925d03a3cc2662d...,maria.miller67@outlook.com,8843b2c0ec23996a07effc6c47ec8716d075d20499b963...,417-25-4912,a0a5f9cec1ea52aaf9c8176d747854c0d1f60780674ac3...
6,Nicholas King,52f245abedecbea4e851326e9fd72cf68ba11a11d13385...,nicholas.king46@outlook.com,03c1a6e2b0058c3775a190b81ba8ceeebdd6ca5493bdd9...,613-23-2503,51280b2e354d80c69fa9bcdf99dbc3b8229a17adb12086...
7,Susan Rivera,de7677e8bcad54099299ff677fc72c66b4dc723b90d31e...,susan.rivera74@gmail.com,67c6d13632c3404f0007ec683b34a767d20044bb57f9b8...,176-97-1864,04adc2ff16245195e17eba7e550fbff5ab83bd5fee56bb...
8,Joseph Lopez,750c74fb22b712d145bd87244c78c405fc7aed67d08499...,joseph.lopez1@gmail.com,a7f7177878859512b05c8a3de55772bfd34459cd3e45f6...,652-70-5530,fb789d550e179a5c56987d22f016090eaf5745279753ff...
9,Michael Mitchell,dcb77bc29d159f435cc748b0eeafe73d030d75a4f9da87...,michael.mitchell42@hotmail.com,f6a2496402be6666d36fc4f471e2e114ba7853288c106b...,100-94-8400,0befb0709774eeb4b98b473b2d4c2f38e9a1bb30854b18...


##### 3.2. Removal of unnecessary PII - Data Minimization

In [17]:
from IPython.display import display

col = "applicant_info.ip_address"

# BEFORE
print("BEFORE: Column exists")
display(df[[col]].head(5))

# DROP
df.drop(columns=[col], inplace=True)

# AFTER
print("AFTER: Column no longer exists")
display(df.head(5))          


BEFORE: Column exists


,applicant_info.ip_address
0,192.168.48.155
1,10.1.102.112
2,10.240.193.250
3,192.168.175.67
4,172.29.125.105


AFTER: Column no longer exists


,_id,spending_behavior,processing_timestamp,applicant_info.gender,applicant_info.date_of_birth,applicant_info.zip_code,financials.annual_income,financials.credit_history_months,financials.debt_to_income,financials.savings_balance,decision.loan_approved,decision.rejection_reason,loan_purpose,decision.interest_rate,decision.approved_amount,financials.annual_salary,notes,full_name_hash,email_hash,ssn_hash
0,app_200,"[{'category': 'Shopping', 'amount': 480}, {'ca...",2024-01-15T00:00:00Z,Male,2001-03-09,10036,73000,23,0.20,31212,False,algorithm_risk_score,NaN,NaN,NaN,NaN,NaN,68ee17cf46b0560352c701bbbdb178c01d4cc368014d18...,116648a7761525746032d0ab323ceb01f50d11f7935164...,2caf30528c21a10e1307b27f9dbbfc312f0c00d46b333e...
1,app_037,"[{'category': 'Rent', 'amount': 608}, {'catego...",NaN,M,1992-03-31,10032,78000,51,0.18,17915,False,algorithm_risk_score,NaN,NaN,NaN,NaN,NaN,4c539f3c4c8794d5d0b7ef2d2c6f3e5ae3e9ab65f1cb54...,c3522c0b54ef9045c73186bcabb53f8e512360ed17e9cc...,2f7da45fefdcfb2c5b4f5b6f1465c22054c36e04fc77c1...
2,app_215,"[{'category': 'Rent', 'amount': 109}]",NaN,Male,1989-10-24,10075,61000,41,0.21,37909,True,NaN,vacation,3.7,59000.0,NaN,NaN,4ad1a6eb65ea21350865c8dbae97a882eb82ee36059784...,b299e7d6a37e183bab209eb8df919652117dd16ed16698...,db120edcee2366a48d6d77c2db8c64c5536b8dc3c3c524...
3,app_024,"[{'category': 'Fitness', 'amount': 575}]",NaN,Male,1983-04-25,10077,103000,70,0.35,0,True,NaN,NaN,4.3,34000.0,NaN,NaN,0b6a308e6a3e28e0a90c5e4212a34d32c0debeccc37408...,6fbd2478748a29faa143392f28d955133e346d09673963...,c835719be02018987096d6e49529a24b1d7e7ab35c84b1...
4,app_184,"[{'category': 'Entertainment', 'amount': 463}]",2024-01-15T00:00:00Z,M,1999-05-21,10080,57000,14,0.23,31763,False,algorithm_risk_score,NaN,NaN,NaN,NaN,NaN,099b981258dca6195cdfe5d42d1d8ab470b12a0926fadc...,f24e7cc1450ee9aa26b833e0593b2420fbfd59b8ad2636...,41c7de40dc49185886e6ecb37346ec9eabce16087b7508...


##### 3.3. Date of Birth Conversion (Anonymization)

In [18]:
import pandas as pd
from IPython.display import display

# Create transformed columns
df["dob"] = pd.to_datetime(df["applicant_info.date_of_birth"], errors="coerce")
df["age"] = pd.Timestamp.now().year - df["dob"].dt.year

# Show before and after
preview = df[[
    "applicant_info.date_of_birth",
    "dob",
    "age"
]].head(10)

display(preview)

# Drop original column after preview
df.drop(columns=["applicant_info.date_of_birth"], inplace=True)


,applicant_info.date_of_birth,dob,age
0,2001-03-09,2001-03-09,25.0
1,1992-03-31,1992-03-31,34.0
2,1989-10-24,1989-10-24,37.0
3,1983-04-25,1983-04-25,43.0
4,1999-05-21,1999-05-21,27.0
5,14/02/1982,NaT,NaN
6,28/01/1990,NaT,NaN
7,1991-10-11,1991-10-11,35.0
8,1990-05-04,1990-05-04,36.0
9,1989-10-10,1989-10-10,37.0


##### 3.4. Additionally: Detailed Spending Removal

Note: Skepticism on Spending Behavior Removal
* Spending behaviour could be useful for evaluating credit risk
* This means it may have a valid business purpose
* But some details might still be too intrusive
* It depends on whether the data is truly necessary and justified

In [19]:
from IPython.display import display
col = "spending_behavior"

# BEFORE
print("BEFORE: Column exists")
display(df[[col]].head(5))

# DROP
df.drop(columns=[col], inplace=True)

# AFTER
print("AFTER: Column no longer exists")
display(df.head(5))   

BEFORE: Column exists


,spending_behavior
0,"[{'category': 'Shopping', 'amount': 480}, {'ca..."
1,"[{'category': 'Rent', 'amount': 608}, {'catego..."
2,"[{'category': 'Rent', 'amount': 109}]"
3,"[{'category': 'Fitness', 'amount': 575}]"
4,"[{'category': 'Entertainment', 'amount': 463}]"


AFTER: Column no longer exists


,_id,processing_timestamp,applicant_info.gender,applicant_info.zip_code,financials.annual_income,financials.credit_history_months,financials.debt_to_income,financials.savings_balance,decision.loan_approved,decision.rejection_reason,loan_purpose,decision.interest_rate,decision.approved_amount,financials.annual_salary,notes,full_name_hash,email_hash,ssn_hash,dob,age
0,app_200,2024-01-15T00:00:00Z,Male,10036,73000,23,0.20,31212,False,algorithm_risk_score,NaN,NaN,NaN,NaN,NaN,68ee17cf46b0560352c701bbbdb178c01d4cc368014d18...,116648a7761525746032d0ab323ceb01f50d11f7935164...,2caf30528c21a10e1307b27f9dbbfc312f0c00d46b333e...,2001-03-09,25.0
1,app_037,NaN,M,10032,78000,51,0.18,17915,False,algorithm_risk_score,NaN,NaN,NaN,NaN,NaN,4c539f3c4c8794d5d0b7ef2d2c6f3e5ae3e9ab65f1cb54...,c3522c0b54ef9045c73186bcabb53f8e512360ed17e9cc...,2f7da45fefdcfb2c5b4f5b6f1465c22054c36e04fc77c1...,1992-03-31,34.0
2,app_215,NaN,Male,10075,61000,41,0.21,37909,True,NaN,vacation,3.7,59000.0,NaN,NaN,4ad1a6eb65ea21350865c8dbae97a882eb82ee36059784...,b299e7d6a37e183bab209eb8df919652117dd16ed16698...,db120edcee2366a48d6d77c2db8c64c5536b8dc3c3c524...,1989-10-24,37.0
3,app_024,NaN,Male,10077,103000,70,0.35,0,True,NaN,NaN,4.3,34000.0,NaN,NaN,0b6a308e6a3e28e0a90c5e4212a34d32c0debeccc37408...,6fbd2478748a29faa143392f28d955133e346d09673963...,c835719be02018987096d6e49529a24b1d7e7ab35c84b1...,1983-04-25,43.0
4,app_184,2024-01-15T00:00:00Z,M,10080,57000,14,0.23,31763,False,algorithm_risk_score,NaN,NaN,NaN,NaN,NaN,099b981258dca6195cdfe5d42d1d8ab470b12a0926fadc...,f24e7cc1450ee9aa26b833e0593b2420fbfd59b8ad2636...,41c7de40dc49185886e6ecb37346ec9eabce16087b7508...,1999-05-21,27.0


##### 3.4. GDPR Mapping - Overall

Lawful Basis:
* Lots of personal data (names, emails, SSNs, IPs, DOB)
* No stated purpose / legal basis
* Credit scoring often uses contractual necessity but must be documented - Under GDPR, the reason for using the data must be clear

Data Minimization:
* Some fields likely unnecessary (IPs, detailed spending?)
* Unnecessary data = higher privacy risk
* Remove/reduce fields to improve compliance

Storage Limitation:
* No retention info (no timestamps/deletion flags)
* Hard to enforce deletion schedules
* Delete/anonymize data when no longer needed